In [ ]:
# =============================================================================
# Moduls
# =============================================================================
import time
import sys
import os
import sqlite3
import subprocess
import json

import utils                         as utils
import database_codes.bch_usdt_1m    as bch_1m
import database_codes.bchusdt_1m_dev as bch_1m_dev

from datetime        import datetime, UTC, timedelta
from IPython.display import clear_output  # Required for clearing the output in Jupyter Notebook

# =============================================================================
# Global variables for logging
# =============================================================================
logs = []

# =============================================================================
# Logging utilities
# =============================================================================
# log_message:
#   - prepend timestamped message to logs list
#   - if section=True add a visual separator block
#   - then call display_logs()
# display_logs:
#   - clear output and print logs (latest first)
# =============================================================================

def log_message(message, section=False):
    # create timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if section:
        logs.insert(0, f"\n{'='*40}\n[{timestamp}] {message}\n{'='*40}")
    else:
        logs.insert(0, f"[{timestamp}] {message}")
    display_logs()

def display_logs():
    # Clear the cell output to refresh logs and print them
    clear_output(wait=True)
    for log in logs:
        print(log)

# =============================================================================
# Sync capture and logging
# =============================================================================
# capture_and_log_sync:
#   - redirect stdout to capture prints from sync function
#   - run bch_1m.sync_bchusdt_1m()
#   - restore stdout and log captured output as a section
# =============================================================================

def capture_and_log_sync():
    from io import StringIO

    # Capture the standard output
    old_stdout     = sys.stdout
    sys.stdout     = captured_output = StringIO()
    try:
        # run the sync from the main module
        bch_1m.sync_bchusdt_1m()
    except Exception as e:
        print(f"Error during sync: {e}")
    finally:
        sys.stdout = old_stdout

    # Log the captured output as a section
    log_message(captured_output.getvalue(), section=True)

# =============================================================================
# Dev table maintenance logic
# =============================================================================
# ensure_dev_table_up_to_date:
#   - ensure the dev table exists and is up to date (23:59 records)
#   - if missing or stale -> call bch_1m_dev.create_dev_data_table(...)
# =============================================================================

def ensure_dev_table_up_to_date(table_name_dev = "bchusdt_1m_dev"):
    # -------------------------------------------------------------------------
    # DB_PATH retrieval via utils config (use repo config for canonical path)
    # -------------------------------------------------------------------------
    cfg       = utils._load_config()
    DB_PATH   = cfg["database"]["db_path"]
    conn      = sqlite3.connect(DB_PATH)
    cursor    = conn.cursor()

    # -------------------------------------------------------------------------
    # Check if dev table exists
    # -------------------------------------------------------------------------
    cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (table_name_dev,))
    exists = cursor.fetchone() is not None

    # -------------------------------------------------------------------------
    # DEV table config data and helpers
    # -------------------------------------------------------------------------
    def dt_from_str(s): return datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=UTC)

    # Table existence check (re-check to be explicit)
    cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (table_name_dev,))
    exists = cursor.fetchone() is not None

    # Target latest date: yesterday at 23:59 UTC (for up-to-date check)
    yesterday_2359 = (datetime.now(UTC) - timedelta(days=1)).replace(hour=23, minute=59, second=0, microsecond=0)

    recreate_dev_table = False

    if not exists:
        # if the table does not exist -> recreate
        recreate_dev_table = True
        log_message(f"Dev table '{table_name_dev}' does not exist. Will be created.", section=True)
    else:
        # Table exists: check latest 23:59 record
        cursor.execute(f"SELECT MAX(open_time) FROM {table_name_dev} WHERE strftime('%H:%M', open_time) = '23:59';")
        latest_dt_str = cursor.fetchone()[0]
        if latest_dt_str is None or dt_from_str(latest_dt_str) < yesterday_2359:
            recreate_dev_table = True
            log_message(f"Dev table '{table_name_dev}' is not up to date. Will be recreated.", section=True)
        else:
            log_message(f"Dev table '{table_name_dev}' is up to date. No action required.", section=True)

    if recreate_dev_table:
        # compute interval for recreation: from epoch start to today 23:59 UTC
        open_time_from = "2017-01-01 00:00:00"
        open_time_to   = datetime.now(UTC).replace(hour=23, minute=59, second=0, microsecond=0).strftime("%Y-%m-%d %H:%M:%S")

        # call dev-table creation in the dev module
        try:
            bch_1m_dev.create_dev_data_table(open_time_from, open_time_to)
            log_message(f"Dev table '{table_name_dev}' created for interval {open_time_from} to {open_time_to}.", section=True)
        except Exception as e:
            log_message(f"Failed to create dev table '{table_name_dev}': {e}", section=True)

    conn.close()

# =============================================================================
# Scheduler logic
# =============================================================================

try:
    while True:
        log_message("Starting sync...", section=True)
        capture_and_log_sync()
        log_message("Waiting for the next cycle...")

        # ---------------------------------------------------------------------
        # Ensure dev table is present and up to date after each cycle
        # ---------------------------------------------------------------------
        ensure_dev_table_up_to_date(table_name_dev="bchusdt_1m_dev")

        time.sleep(30)  # Wait 30 seconds
except KeyboardInterrupt:
    log_message("Scheduler interrupted. Exiting...", section=True)


[2025-10-28 08:16:39] Failed to create dev table 'bchusdt_1m_dev': 'table_name'

[2025-10-28 08:16:39] Dev table 'bchusdt_1m_dev' does not exist. Will be created.
[2025-10-28 08:16:39] Waiting for the next cycle...

[2025-10-28 08:16:39] Attempted to insert rows: 2
SQLite reported total changes in this connection: 2
Range inserted (source frame): 2025-10-28 09:15 -> 2025-10-28 09:16


[2025-10-28 08:16:37] Starting sync...
